# NeuroGuard — Phase 3: always-empty baseline and BCE U-Net
This project is a prototype and is not medical advice.

Run numbered sections in order in **browser Colab with a T4 GPU**. Real execution
is performed by you. No metrics are pre-filled. Keep this notebook and all outputs
private: the data and manifest contain patient identifiers. Do not publish outputs
without privacy/licence review. Source edits belong in the local repository.

Inputs: reviewed full commit SHA, original Phase 2 manifest hash, dataset terms,
run ID/resume choice, and smoke-test review before full training. No Tversky,
test predictions, split generation or Gradio. Phase 2 evidence is read-only.
After a reset rerun sections 1–10 with the SAME SHA/config/run ID, then resume in
11; completed epochs count towards the original 30-epoch budget.


## 1. Verify Colab and GPU
Select Runtime → Change runtime type → T4 GPU. Preserve the installed CUDA PyTorch.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, re
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
from google.colab import drive
import torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime; do not silently start full training on CPU')
print('GPU:', torch.cuda.get_device_name(0))


## 2. Enter the reviewed full GitHub commit SHA
After an approved push, open this notebook at that exact commit. Opening a notebook does not clone src/.


In [ ]:
REPOSITORY = 'https://github.com/hrb31/NeuroGuard.git'
REVISION = input('Full reviewed Phase 3 Git commit SHA: ').strip().lower()
if not re.fullmatch(r'[0-9a-f]{40}', REVISION):
    raise ValueError('Full 40-character Git SHA required')
PRIVATE_REPOSITORY = False  # Set True only if authentication is required.
REPO = Path('/content/NeuroGuard')


## 3. Clone and verify source revision
For a private repository, grant Colab Secret GITHUB_TOKEN read-only access or use
the hidden prompt. Credentials never go in URLs/source. A stale or edited checkout
stops execution: save outputs and use a clean runtime.


In [ ]:
import tempfile
from getpass import getpass
env = os.environ.copy()
env['GIT_TERMINAL_PROMPT'] = '0'
with tempfile.TemporaryDirectory() as auth_dir:
    if PRIVATE_REPOSITORY:
        try:
            from google.colab import userdata
            token = userdata.get('GITHUB_TOKEN')
        except Exception:
            token = getpass('Read-only GitHub token (hidden): ')
        if not token:
            raise ValueError('Authentication required')
        helper = Path(auth_dir) / 'askpass.py'
        helper.write_text('#!/usr/bin/env python3\nimport os,sys\nprint("x-access-token" if "Username" in sys.argv[1] else os.environ["NG_GITHUB_TOKEN"])\n')
        helper.chmod(0o700)
        env.update(GIT_ASKPASS=str(helper), NG_GITHUB_TOKEN=token)
        del token
    try:
        if not REPO.exists():
            subprocess.run(['git', 'clone', '--quiet', '--no-checkout', REPOSITORY, str(REPO)], env=env, check=True)
            subprocess.run(['git', '-C', str(REPO), 'checkout', '--quiet', '--detach', REVISION], env=env, check=True)
        actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
        if actual != REVISION:
            raise RuntimeError('Existing checkout differs. Save outputs, delete/reset the Colab runtime, and rerun. No local edits were overwritten.')
        if subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip():
            raise RuntimeError('Checkout has local changes. Use a clean runtime.')
    finally:
        env.pop('NG_GITHUB_TOKEN', None)
        env.pop('GIT_ASKPASS', None)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('Executing source commit:', actual)


## 4. Install only supporting dependencies; run synthetic tests
No torch replacement. If already-imported supporting packages change, restart and
rerun at the same SHA. Tests use synthetic TIFFs, not private patient data. The
Phase 2 notebook also remains usable with audit-only dependencies.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-training.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=True)


## 5. Mount Drive and choose experiment identity
Keep the existing Phase 2 folder. Choose one simple run ID (for example bce-seed42-01).
Enter RESUME only for the same interrupted run. New run directories cannot be
overwritten. Preflight/smoke outputs go in a separate timestamped directory.


In [ ]:
drive.mount('/content/drive')
from datetime import datetime, timezone
from src.data import write_json, digest
from src.training import seed_everything
PROJECT = Path('/content/drive/MyDrive/NeuroGuard')
PHASE2 = PROJECT / 'dataset-v2-seed42'
RUN_ID = input('Run ID (letters, digits, hyphen, underscore): ').strip()
if not re.fullmatch(r'[A-Za-z0-9_-]{1,60}', RUN_ID):
    raise ValueError('Invalid run ID')
RESUME = input('Type RESUME for an existing interrupted run; NEW for a new run: ').strip()
if RESUME not in ('RESUME', 'NEW'):
    raise ValueError('Choose RESUME or NEW explicitly')
RUN = PROJECT / 'experiments' / 'phase3-bce' / RUN_ID
PREFLIGHT = PROJECT / 'experiments' / 'phase3-bce' / 'preflight' / RUN_ID / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
PREFLIGHT.mkdir(parents=True, exist_ok=False)
config = json.loads((REPO / 'configs/bce.json').read_text())
seed_everything(config['seed'])
environment_text = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
environment = {'python': sys.version, 'torch': str(torch.__version__),
    'cuda': torch.version.cuda, 'cudnn': torch.backends.cudnn.version(),
    'gpu': torch.cuda.get_device_name(0), 'packages': environment_text,
    'deterministic_algorithms': torch.are_deterministic_algorithms_enabled(),
    'cublas_workspace_config': os.environ['CUBLAS_WORKSPACE_CONFIG']}
write_json(PREFLIGHT/'environment.json', environment)
(PREFLIGHT/'environment.txt').write_text(environment_text)
print('BCE run:', RUN)
print(json.dumps(config, indent=2))


## 6. Load and verify the existing frozen manifest (never regenerate)
Paste the **Frozen manifest SHA256 printed in Phase 2 section 9** or read it from
the original private evidence archive. This is the canonical JSON digest stored
in manifest_sha256, not the raw file-byte hash. A wrong/missing identity stops.
This cell checks the original split counts supplied in the Phase 3 request and
loads the original training-derived numeric size cutoffs without recomputing them.


In [ ]:
from src.dataset import load_manifest, verify_dataset, MRIDataset, make_loader, file_sha256
EXPECTED_MANIFEST_HASH = input('Original Phase 2 Frozen manifest SHA256: ').strip().lower()
if not re.fullmatch(r'[0-9a-f]{64}', EXPECTED_MANIFEST_HASH):
    raise ValueError('Original 64-character manifest identity is required')
manifest_path = PHASE2 / 'split_manifest.json'
manifest = load_manifest(manifest_path, EXPECTED_MANIFEST_HASH)
manifest_file_sha256 = file_sha256(manifest_path)
expected_counts = {'train': (77, 2719, 934), 'validation': (16, 621, 209), 'test': (17, 589, 230)}
for subset, expected in expected_counts.items():
    observed = manifest['summary'][subset]
    if tuple(observed[k] for k in ('patients', 'slices', 'positive_slices')) != expected:
        raise ValueError('Frozen split differs from the supplied verified Phase 2 counts')
print('Manifest identity:', manifest['manifest_sha256'])
print('Dataset fingerprint:', manifest['dataset_fingerprint'])
print('Original training size cutoffs:', manifest['size_cutoffs'])
print('Frozen counts:', json.dumps(manifest['summary'], indent=2))


## 7. Access version-2 data in Colab; verify the canonical root
Read the [Kaggle release](https://www.kaggle.com/datasets/mateuszbuda/lgg-mri-segmentation),
[CC BY-NC-SA 4.0 terms](https://creativecommons.org/licenses/by-nc-sa/4.0/) and
upstream terms. Cached version-2 data is reused when available. If login is needed,
grant the Colab KAGGLE_API_TOKEN secret access or run kagglehub.login() with its
hidden prompt. Never send credentials back. No Windows dataset download.

Reuse the Phase 2 root resolver, including complete duplicate-directory hash
comparison. Verify every saved pair's bytes against the original manifest. This is
an identity preflight, not a new Phase 2 audit, new split or new size calibration.
Test TIFFs are checked for integrity only; only development pairs are decoded.


In [ ]:
from src.acquisition import download_dataset, resolve_dataset_root, get_metadata
release = get_metadata()
if release['metadata']['licenseName'] != 'CC BY-NC-SA 4.0' or release['metadata']['currentVersionNumber'] != 2:
    raise RuntimeError('Release metadata changed; stop for review')
write_json(PREFLIGHT/'dataset_release.json', release)
terms_accepted = input('After reviewing the dataset/upstream terms, type ACCEPT: ') == 'ACCEPT'
DOWNLOAD_ROOT = download_dataset(terms_accepted=terms_accepted)
DATA_ROOT, root_report = resolve_dataset_root(DOWNLOAD_ROOT)
verify_dataset(DATA_ROOT, manifest)
write_json(PREFLIGHT/'root_resolution.json', root_report)
print('Canonical root verified; no assignments regenerated or modified.')


## 8. Verify development loaders and model
Training only has seeded paired augmentation; validation is deterministic. All
three stored MRI channels are used: pre-contrast, FLAIR, post-contrast (FLAIR may
substitute missing sequences per release README). No test Dataset is constructed.


In [ ]:
from src.model import UNet
from src.training import evaluate, fit
train_data = MRIDataset(DATA_ROOT, manifest, 'train', **config['preprocessing'], seed=config['seed'])
val_data = MRIDataset(DATA_ROOT, manifest, 'validation', **config['preprocessing'], seed=config['seed'])
train_patients = {r['patient'] for r in train_data.records}
val_patients = {r['patient'] for r in val_data.records}
assert not train_patients & val_patients
batch = next(iter(make_loader(train_data, config)))
assert batch['image'].shape[1:] == (3, 256, 256)
assert batch['mask'].shape[1:] == (1, 256, 256)
assert set(batch['mask'].unique().tolist()) <= {0, 1}
device = torch.device('cuda')
seed_everything(config['seed'])
check_model = UNet(**config['model']).to(device)
with torch.no_grad():
    assert check_model(batch['image'][:2].to(device)).shape == (2, 1, 256, 256)
parameter_count = sum(p.numel() for p in check_model.parameters())
print('Parameters:', parameter_count, 'train/validation slices:', len(train_data), len(val_data))
del check_model, batch
torch.cuda.empty_cache()


## 9. Compute Model 0 on validation only
The predictor returns exactly zero, with no fitting. Positive-slice Dice/recall
and conventional precision are zero; pooled precision has a zero denominator and
is NA (JSON null). Patient/slice denominators and frozen-size groups are reported.
High pixel accuracy can simply reflect background dominance despite zero recall.
These numbers become findings only when this cell actually executes.


In [ ]:
baseline = evaluate(None, make_loader(val_data, config), device, manifest['size_cutoffs'])['reports'][.5]
write_json(PREFLIGHT/'always_empty_validation.json', baseline)
print(json.dumps(baseline, indent=2))


## 10. Train-only tiny-subset smoke test and review gate
Run 100 updates on four fixed, augmented positive **training** examples. Check
finite loss/gradients, loss decrease and exact checkpoint reload. Tiny training
Dice is only a debugging observation, not validation performance. Inspect smoke
overlays: decreasing BCE alone does not establish successful foreground fitting.
If loss does not decrease or the check fails, stop and return the error. After
review set SMOKE_REVIEWED=True below before section 11. The definitive fit creates
a fresh seed42 model/optimiser; smoke weights are never used for full training.


In [ ]:
from src.smoke import tiny_overfit
smoke = tiny_overfit(train_data, config, PREFLIGHT/'smoke', steps=100)
write_json(PREFLIGHT/'smoke.json', smoke)
print(json.dumps(smoke, indent=2))
from IPython.display import display, Image
display(Image(filename=str(PREFLIGHT/'smoke'/'smoke_debug.png')))


In [ ]:
SMOKE_REVIEWED = False  # USER INPUT: set True after reviewing the outputs above, then run this small cell.


## 11. Freeze provenance and train/resume BCE
USER INPUT: type TRAIN after smoke review. This confirms the printed frozen
defaults, including research FP guardrails (10% absolute/5pp comparison). Max30
epochs, patience7, batch16, Adam .001. No schedule, mixed precision or class weights.
An OOM must stop: do not silently lower batch size. A batch8 change requires a
recorded decision for BOTH definitive comparison arms before a fresh experiment.

Checkpoints persist every completed epoch. A disconnect replays only the unfinished
epoch. Resume demands identical config, SHA, manifest and package/GPU environment.
No new training budget or automatic fresh run. best.pt is exported after completion;
the selected best state is also inside every training-state checkpoint.


In [ ]:
if not SMOKE_REVIEWED or not smoke['loss_decreased']:
    raise RuntimeError('Review the smoke outputs and set SMOKE_REVIEWED=True in section 10')
if input('Type TRAIN to run the frozen BCE experiment: ') != 'TRAIN':
    raise RuntimeError('Training not confirmed')
identity = {'config': config, 'commit': REVISION, 'environment': environment,
    'manifest_sha256': manifest['manifest_sha256'], 'manifest_file_sha256': manifest_file_sha256,
    'dataset_fingerprint': manifest['dataset_fingerprint'], 'size_cutoffs': manifest['size_cutoffs'],
    'parameter_count': parameter_count, 'channel_mapping': ['pre-contrast', 'FLAIR', 'post-contrast']}
write_json(PREFLIGHT/'frozen_protocol.json', identity)
try:
    model, history = fit(train_data, val_data, config, RUN, identity, resume=RESUME == 'RESUME')
except BaseException as error:
    write_json(PREFLIGHT/'training_failure.json', {'type': type(error).__name__,
        'message': str(error), 'commit': REVISION, 'run': str(RUN),
        'manifest_sha256': manifest['manifest_sha256'], 'test_evaluated': False})
    raise
print('Completed epochs:', len(history))


## 12. Validation metrics, threshold selection and plots
The best checkpoint was selected at .5. Perform the single declared validation
search .10:.05:.90, maximise patient-macro positive-slice Dice subject to empty-mask
FP <=10%, ties by lower FP then higher threshold. Fallback .5 if none qualifies.
Report .5 and selected threshold separately, with eligible counts and frozen-size
groups. No component cleanup or test tuning. Plots use stored FLAIR channel 1.


In [ ]:
from src.metrics import select_threshold
from src.reporting import save_curves, save_predictions
validation = evaluate(model, make_loader(val_data, config), device, manifest['size_cutoffs'],
                      config['thresholds'], torch.nn.BCEWithLogitsLoss())
selection = select_threshold(validation['reports'], config['empty_fp_limit'])
selected = selection['threshold']
reports = {'default_threshold': .5, 'default': validation['reports'][.5],
           'selection': selection, 'selected': validation['reports'][selected],
           'threshold_search': validation['reports'], 'validation_loss': validation['loss']}
write_json(RUN/'validation_reports.json', reports)
save_curves(history, RUN)
save_predictions(model, val_data, device, selected, RUN/'figures')
print(json.dumps({'default': reports['default'], 'selection': selection, 'selected': reports['selected']}, indent=2))
from IPython.display import display, Image
display(Image(filename=str(RUN/'training_curves.png')))
for path in sorted((RUN/'figures').glob('*.png')):
    display(Image(filename=str(path)))


## 13. Save reproducibility records and verify frozen evidence unchanged
The two rotating state files plus latest.json provide resume automatically; keep
the entire run folder. Load only your own trusted checkpoint. Package/GPU versions,
config, optimiser/RNG, source SHA, manifest identities and histories accompany it.
Raw data stays in Colab; no original Phase 2 evidence is overwritten.


In [ ]:
assert file_sha256(manifest_path) == manifest_file_sha256, 'Frozen manifest changed: STOP'
(RUN/'environment.txt').write_text(environment_text)
write_json(RUN/'config.json', config)
write_json(RUN/'always_empty_validation.json', baseline)
write_json(RUN/'smoke_debug_only.json', smoke)
best_epoch = max(history, key=lambda r: r['validation']['positive_patient_macro']['dice'])['epoch']
summary = {'stage': 'Phase 3 BCE validation only', 'commit': REVISION,
    'manifest_sha256': manifest['manifest_sha256'], 'parameter_count': parameter_count,
    'completed_epochs': len(history), 'best_epoch': best_epoch,
    'always_empty': baseline, 'bce_default': reports['default'],
    'selection': selection, 'bce_selected': reports['selected'],
    'test_evaluated': False, 'phase4_started': False,
    'training_seconds': sum(r['seconds'] for r in history)}
write_json(RUN/'review_summary.json', summary)
write_json(RUN/'artifact_hashes.json', {p.relative_to(RUN).as_posix(): file_sha256(p)
    for p in sorted(RUN.rglob('*')) if p.is_file() and p.name != 'artifact_hashes.json'})
print('Persistent private run:', RUN)
print('Best checkpoint SHA256:', file_sha256(RUN/'best.pt'))


## 14. Return results for review; stop before Phase 4
Copy the JSON below, synthetic pytest summary, smoke output, training curve and
representative overlays back to Codex. Do not send the manifest, patient IDs,
tokens, full checkpoint or raw TIFFs. Record any errors/OOMs honestly. This is
validation evidence; final test evaluation remains locked for Phase 5.

To resume after reset: reopen the SAME full SHA, repeat 1–10, choose the SAME run
ID and RESUME in section 5, review smoke, then run 11–14. If identity differs,
stop and record the discrepancy instead of modifying checkpoints or the split.
References: [U-Net paper](https://arxiv.org/abs/1505.04597),
[PyTorch reproducibility](https://docs.pytorch.org/docs/stable/notes/randomness.html),
[BCEWithLogitsLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html),
[checkpoint guide](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html).
Prepared with OpenAI Codex assistance; the student reviews, executes and interprets.


In [ ]:
print((RUN/'review_summary.json').read_text())
